# Autoregressive 模型评测

加载 canonical artifact，比较随机生成质量、surprisal 和搜索覆盖率。

## 加载模型和测试集

In [ ]:
import gc
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from scripts.pipeline import get_device
from scripts.monitoring import TensorBoardMonitor
from scripts.evaluation import (
    coverage_curve,
    evaluation_artifact_paths,
    evaluate_random_generation_with_coverage,
    plot_coverage_curve,
    plot_efficiency_curve,
    plot_generation_quality,
    plot_random_coverage_curve,
    plot_random_efficiency_curve,
    plot_surprisal_boxplot,
    plot_surprisal_histogram,
    read_test_passwords,
    save_evaluation_summary,
    summarize_surprisal,
    save_coverage_npz,
    save_surprisal_npz
)
from scripts.inference import (
    best_first_search,
    load_generation_candidates,
    load_inference_model,
    score_passwords,
)
from scripts.tokenizer import CharTokenizer

device = get_device()
data_dir = Path("data/processed")
tokenizer_path = data_dir / "tokenizer.json"
evaluation_root = Path("output/evaluation")
evaluation_tier = "medium"  # 可切换为 low 或 high，每次运行只评测一个神经模型档位。
include_baseline = False  # Bigram 已有独立结果时无需重复执行大规模评测。
tier_label = evaluation_tier.capitalize()
model_specs = [
    (evaluation_tier, "mlp", f"MLP · {tier_label}"),
    (evaluation_tier, "gru", f"GRU · {tier_label}"),
    (evaluation_tier, "tcn", f"TCN · {tier_label}"),
    (evaluation_tier, "transformer", f"Transformer · {tier_label}"),
]
if include_baseline:
    model_specs.insert(0, ("baseline", "bigram", "Bigram · Baseline"))
comparison_dir = evaluation_root / evaluation_tier / "comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
test_passwords = read_test_passwords(data_dir / "test.txt", seed=42)
tokenizer = CharTokenizer.from_json(tokenizer_path)
models = {}
artifact_paths = {}
for tier, model_type, display_name in model_specs:
    model_dir = (Path("output") / model_type if tier == "baseline"
                 else Path("output") / tier / model_type)
    model, _ = load_inference_model(
        model_dir / "model.pt", tokenizer_path, model_dir / "inference.json", device=device
    )
    models[display_name] = model
    artifact_paths[display_name] = evaluation_artifact_paths(
        evaluation_root, tier, model_type
    )
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_log_dir = (Path("/root/tf-logs/passmini/evaluation")
                       / evaluation_tier / run_id)
monitor = TensorBoardMonitor.create(tensorboard_log_dir)
print(f"TensorBoard logs: {tensorboard_log_dir}")

## Surprisal 分布

In [ ]:
surprisal = {name: score_passwords(model, tokenizer, test_passwords,
                                   batch_size=10_000, verbose=True,
                                   progress_callback=monitor.progress_callback(f"Surprisal/{name}"))
             for name, model in models.items()}
surprisal_per_token = {name: [sp / (len(pw)+1)
                              for sp, pw in zip(values, test_passwords)]
                       for name, values in surprisal.items()}
summaries = {name: summarize_surprisal(test_passwords, values)
             for name, values in surprisal.items()}
for name in models:
    save_surprisal_npz(
        artifact_paths[name].surprisal,
        surprisal[name],
        surprisal_per_token[name],
        max_points=100_000,
    )
    save_evaluation_summary(artifact_paths[name].summary, surprisal=summaries[name])
    monitor.log_metrics(
        f"Surprisal/{name}",
        {"mean_bits_per_token": summaries[name].bits_per_token.mean},
        len(test_passwords),
    )
monitor.flush()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_surprisal_histogram(surprisal, ax=axes[0, 0])
plot_surprisal_boxplot(surprisal, ax=axes[0, 1])
plot_surprisal_histogram(surprisal_per_token, ax=axes[1, 0], value_label="Surprisal (bits/token)")
plot_surprisal_boxplot(surprisal_per_token, ax=axes[1, 1], value_label="Surprisal (bits/token)")
fig.tight_layout()
fig.savefig(comparison_dir / "surprisal.png")

## 随机生成覆盖率

In [ ]:
random_num_samples = 100_000_000
random_batch_sizes = {
    "bigram": 500_000,
    "mlp": 100_000,
    "gru": 100_000,
    "tcn": 20_000,
    "transformer": 20_000,
}

def report_cuda_memory(name, step):
    if device.type != "cuda":
        return
    torch.cuda.synchronize(device)
    gib = 1024 ** 3
    metrics = {
        "allocated_gib": torch.cuda.memory_allocated(device) / gib,
        "reserved_gib": torch.cuda.memory_reserved(device) / gib,
        "peak_allocated_gib": torch.cuda.max_memory_allocated(device) / gib,
    }
    print(f"CUDA {name}: " + ", ".join(f"{key}={value:.2f}" for key, value in metrics.items()))
    monitor.log_metrics(f"CUDA/{name}", metrics, step, flush=True)

random_results = {}
random_quality = {}
for name, model in models.items():
    if device.type == "cuda":
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        report_cuda_memory(name, 0)
    batch_size = random_batch_sizes[model.model_type]
    print(f"Random generation: {name}, batch_size={batch_size:,}")
    evaluation = evaluate_random_generation_with_coverage(
        model, test_passwords, num_samples=random_num_samples, batch_size=batch_size,
        max_length=12, generator=torch.Generator(model.device).manual_seed(2026),
        checkpoint_step=100, verbose=True,
        progress_callback=monitor.progress_callback(f"RandomGeneration/{name}"),
    )
    random_results[name] = evaluation.coverage
    random_quality[name] = evaluation.quality
    save_coverage_npz(
        artifact_paths[name].random_coverage,
        evaluation.coverage,
        test_size=len(test_passwords),
    )
    save_evaluation_summary(
        artifact_paths[name].summary, generation_quality=evaluation.quality
    )
    monitor.log_metrics(
        f"RandomGeneration/{name}",
        {"legal_rate": evaluation.quality.legal_rate,
         "legal_unique_rate": evaluation.quality.legal_unique_rate,
         "final_coverage": evaluation.coverage[-1].coverage},
        evaluation.quality.total_samples,
    )
    monitor.flush()
    if device.type == "cuda":
        report_cuda_memory(name, random_num_samples)
    del evaluation
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
        report_cuda_memory(name, random_num_samples + 1)

In [ ]:
random_quality

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_generation_quality(random_quality, ax=axes[0])
plot_random_coverage_curve(random_results, ax=axes[1])
plot_random_efficiency_curve(random_results, ax=axes[2])
fig.tight_layout()
fig.savefig(comparison_dir / "random_generation.png")

## Best-first 搜索覆盖率

In [ ]:
load_generated = False
best_first_expansion_batch_size = 64  # 大于 1 启用近似批量弹堆，需先比较速度与候选质量。
best_first_use_cache = False  # 大规模搜索不让每个堆节点持有 TCN 历史或 Transformer KV cache。
search_results = {}
for name, model in models.items():
    path = artifact_paths[name].best_first_candidates
    candidates = (
        load_generation_candidates(path) if path.exists() and load_generated else best_first_search(
            model, tokenizer, num_candidates=100_0000, max_length=12,
            node_top_k=5, depth_beam_width=10_0000,
            expansion_batch_size=best_first_expansion_batch_size,
            use_cache=best_first_use_cache,
            save_path=path, verbose=True,
            progress_callback=monitor.progress_callback(f"BestFirst/{name}"),
        )
    )
    points = coverage_curve(candidates, test_passwords, checkpoint_step=100)
    search_results[name] = points
    save_coverage_npz(
        artifact_paths[name].best_first_coverage,
        points,
        test_size=len(test_passwords),
    )
    monitor.log_metrics(
        f"BestFirst/{name}",
        {"final_coverage": points[-1].coverage,
         "final_efficiency": points[-1].efficiency},
        len(candidates),
    )
    monitor.flush()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_coverage_curve(search_results, ax=axes[0])
plot_efficiency_curve(search_results, ax=axes[1])
fig.tight_layout()
fig.savefig(comparison_dir / "search_coverage.png")
monitor.close()